# Create Silver Schema and Tables

## Setup: Set Your Catalog

**IMPORTANT:** Before running any queries, set your assigned catalog name.


In [0]:
-- TODO: Replace with your assigned catalog name
USE CATALOG redhatmatt;


⚠️ **CI Compatibility Warning**

The CI/CD tests run your DDL against a local Spark environment that is stricter than Databricks. To avoid test failures:

- **Do not use `CREATE OR REPLACE TABLE`** — use `CREATE TABLE` or `CREATE TABLE IF NOT EXISTS` instead

A script to create the silver layer from scratch. We need to build:

* The `silver` schema (if it doesn't already exist)
* 6 tables (if they don't already exist):
    * `categories`
    * `stores`
    * `books`
    * `customers`
    * `orders`
    * `order_items`

In [0]:
-- TODO: Create the silver schema. Use IF NOT EXISTS.
create schema if not exists silver;


**`silver.categories`** — columns to define:
| Column | Type |
|---|---|
| `category_id` | STRING |
| `category_name` | STRING |
| `parent_category_id` | STRING |

In [0]:
-- TODO: Create the silver.categories table
create table if not exists silver.categories (
    category_id string COMMENT "Natural key"
    , category_name string
    , parent_category_id string COMMENT "FK to self; empty string for top level"
) USING DELTA;


**`silver.stores`** — columns to define:
| Column | Type |
|---|---|
| `store_nbr` | STRING |
| `name` | STRING |
| `address` | STRING |
| `city` | STRING |
| `state` | STRING |
| `zip` | STRING |

In [0]:
-- TODO: Create the silver.stores table
create table if not exists silver.stores (
    store_nbr STRING
    , name string
    , address string
    , city string
    , state string
    , zip string
) USING DELTA;


**`silver.books`** — columns to define:
| Column | Type |
|---|---|
| `isbn` | STRING |
| `title` | STRING |
| `author` | STRING |
| `category_id` | STRING |

In [0]:
-- TODO: Create the silver.books table
CREATE TABLE IF NOT EXISTS silver.books (
    isbn string COMMENT "Natural key"
    , title string
    , author STRING
    , category_id STRING COMMENT "FK to silver.categories (leaf only)"
) USING DELTA;


**`silver.customers`** — columns to define:
| Column | Type |
|---|---|
| `email` | STRING |
| `name` | STRING |
| `address` | STRING |
| `city` | STRING |
| `state` | STRING |
| `zip` | STRING |

In [0]:
-- TODO: Create the silver.customers table
CREATE TABLE IF NOT EXISTS silver.customers (
    email STRING COMMENT "Natural key"
    , name STRING COMMENT "From most recent order"
    , address STRING COMMENT "From most recent order"
    , city STRING COMMENT "From most recent order"
    , state STRING COMMENT "From most recent order"
    , zip STRING COMMENT "From most recent order"
) USING DELTA;


**`silver.orders`** — columns to define:
| Column | Type |
|---|---|
| `order_id` | STRING |
| `order_channel` | STRING |
| `order_datetime` | TIMESTAMP |
| `customer_email` | STRING |
| `store_nbr` | STRING |
| `payment_method` | STRING |
| `total_amount` | DECIMAL(10, 2) |
| `cashier_name` | STRING |

In [0]:
-- TODO: Create the silver.orders table
CREATE TABLE IF NOT EXISTS silver.orders
(
    order_id STRING COMMENT "From source bronze tables."
    , order_channel STRING COMMENT "online or in-store"
    , order_datetime TIMESTAMP COMMENT "From order_timestamp or transaction_timestamp"
    , customer_email STRING COMMENT "Coalesce to in-store for anonymous in store orders"
    , store_nbr STRING COMMENT "Coalesce to online for online orders"
    , payment_method STRING
    , total_amount DECIMAL(10,2) COMMENT "Kept for data quality cross-checks"
    , cashier_name STRING COMMENT "Nullable (instore only)"
) USING DELTA COMMENT "Unified orders from online + instore";


**`silver.order_items`** — columns to define:
| Column | Type |
|---|---|
| `order_id` | STRING |
| `order_channel` | STRING |
| `isbn` | STRING |
| `quantity` | INT |
| `unit_price` | DECIMAL(10, 2) |

In [0]:
-- TODO: Create the silver.order_items table
CREATE TABLE IF NOT EXISTS silver.order_items (
    order_id STRING COMMENT "FK to silver.orders"
    , order_channel STRING COMMENT "FK to silver.orders"
    , isbn STRING COMMENT "FK to books"
    , quantity INT
    , unit_price DECIMAL(10,2)
)
